In [1]:
import torch
import pandas as pd
from tqdm import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer

/Users/marconatale/Documents/GitHub/Magistrale/MNLP/HW2/.venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(
/Users/marconatale/Documents/GitHub/Magistrale/MNLP/HW2/.venv/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
device = torch.device("mps") if torch.backends.mps.is_available() else \
        torch.device("cuda") if torch.cuda.is_available() else \
        torch.device("cpu")

In [3]:
print(f"Using device: {device}")

Using device: mps


# Loading the Qwen3 Model

In [4]:
model_name = "Qwen/Qwen3-1.7B"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype="auto",
    device_map="auto"
)

Loading checkpoint shards: 100%|██████████| 2/2 [00:03<00:00,  1.75s/it]


# Reading the Dataset

We'll load the CSV file containing Old Italian sentences that need to be translated to Modern Italian

In [5]:
# Load dataset
df = pd.read_csv("dataset.csv")

# Display basic information
print("DataFrame shape:", df.shape)
print("\nFirst 5 rows:")
df.head()


DataFrame shape: (97, 4)

First 5 rows:


,Author,Date,Region,Sentence
0,Brunetto Latini,1260-61,fior.,quella guerra ben fatta l' opera perché etc. E...
1,Bono Giamboni,1292,fior.,"crudele, e di tutte le colpe pigli vendetta, c..."
2,Valerio Massimo (red. V1,1336,fior.,Non d' altra forza d' animo fue ornato Ponzio ...
3,Lucano volg. (ed. Marinoni),1330/40,prat.,Se questo piace a tutti e se 'l tempo hae biso...
4,Brunetto Latini,1260-61,fior.,Officio di questa arte pare che sia dicere app...


# Translating the Sentences (Zero shot & Instruction-Style)

In [6]:
""" prompts = [
    "You are an expert in historical Italian linguistics. Translate this archaic Italian sentence to modern Italian:\n{sentence}\nONLY provide the translation in your response, nothing else.", # prompt with role
    "Translate this archaic Italian sentence to modern Italian:\n{sentence}\nONLY provide the translation in your response, nothing else.", # prompt without role
    "Consider the following historical context:\nAuthor: {author}\nTime period: {date}\nGeographical region: {region}\n\nHere is an archaic Italian sentence to translate to modern Italian:\n{sentence}\nONLY provide the translation in your response, nothing else.", # prompt with context
    "Sei un esperto di linguistica storica italiana. Traduci questa frase in italiano antico (1300) in italiano moderno:\n{sentence}\nRispondi con SOLO la traduzione", # prompt in italian
] """

prompts = [
    [
        {"role": "system", "content": "You are an expert in historical Italian linguistics. Only provide the translation in your response, nothing else."},
        {"role": "user", "content": "Translate this archaic Italian sentence to modern Italian:\n{sentence}"}
    ],
    [
        {"role": "system", "content": "Only provide the translation in your response, nothing else."},
        {"role": "user", "content": "Translate this archaic Italian sentence to modern Italian:\n{sentence}"}
    ],
    [
        {"role": "system", "content": "Only provide the translation in your response, nothing else."},
        {"role": "user", "content": "Consider the following historical context:\nAuthor: {author}\nTime period: {date}\nGeographical region: {region}\n\nHere is an archaic Italian sentence to translate to modern Italian:\n{sentence}"}
    ],
    [
        {"role": "system", "content": "Sei un esperto di linguistica storica italiana. Rispondi con SOLO la traduzione."},
        {"role": "user", "content": "Traduci questa frase in italiano antico (1300) in italiano moderno:\n{sentence}"}
    ]
]


In [5]:
def translate(df, prompt):
    translations = []
    for _, row in tqdm(df.iterrows(), total=df.shape[0]):
        # Create a copy of the prompt messages to avoid modifying the original
        formatted_prompt = []
        for message in prompt:
            message_copy = message.copy()
            # Replace placeholders in the content
            if "{sentence}" in message_copy["content"]:
                if any(placeholder in message_copy["content"] for placeholder in ["{author}", "{date}", "{region}"]):
                    message_copy["content"] = message_copy["content"].format(
                        sentence=row['Sentence'],
                        author=row['Author'],
                        date=row['Date'],
                        region=row['Region']
                    )
                else:
                    message_copy["content"] = message_copy["content"].format(sentence=row['Sentence'])
            formatted_prompt.append(message_copy)
        
        text = tokenizer.apply_chat_template(
            formatted_prompt,
            tokenize=False,
            add_generation_prompt=True,
            enable_thinking=True # Switches between thinking and non-thinking modes. Default is True.
        )
        model_inputs = tokenizer([text], return_tensors="pt").to(model.device)

        # conduct text completion
        generated_ids = model.generate(
            **model_inputs,
            max_new_tokens=32768
        )
        output_ids = generated_ids[0][len(model_inputs.input_ids[0]):].tolist()
        # parsing thinking content
        try:
            # rindex finding 151668 (</think>)
            index = len(output_ids) - output_ids[::-1].index(151668)
        except ValueError:
            index = 0

        # thinking_content = tokenizer.decode(output_ids[:index], skip_special_tokens=True).strip("\n")
        translation = tokenizer.decode(output_ids[index:], skip_special_tokens=True).strip("\n")
        translations.append(translation)
    return translations

In [8]:
test_df = df.head(3)  # Using just the first 3 rows for testing

# Create a copy of the test dataframe to store results
test_results = test_df.copy()

print("Testing translation with each prompt...")
for idx, prompt in enumerate(prompts):
    test_col_name = f"Test_Modern_Italian_{idx}"
    test_results[test_col_name] = translate(test_df, prompt)
    
    # Display sample results
    print("\nSample results:")
    for i, row in test_results.iterrows():
        print(f"Original: {row['Sentence'][:50]}...")
        print(f"Translation: {row[test_col_name][:50]}...")
        print("---")

Testing translation with each prompt...


100%|██████████| 3/3 [00:14<00:00,  4.85s/it]



Sample results:
Original: quella guerra ben fatta l' opera perché etc. Et da...
Translation: Quella guerra ben fatta l'opera perché etc. E dall...
---
Original: crudele, e di tutte le colpe pigli vendetta, come ...
Translation: crudele, e di tutte le colpe pigli vendetta, come ...
---
Original: Non d' altra forza d' animo fue ornato Ponzio Aufi...
Translation: Non d' altra forza d' animo fu ornato Ponzio Aufid...
---


100%|██████████| 3/3 [00:51<00:00, 17.06s/it]



Sample results:
Original: quella guerra ben fatta l' opera perché etc. Et da...
Translation: Quella guerra ben fatta l'opera perché etc. E dall...
---
Original: crudele, e di tutte le colpe pigli vendetta, come ...
Translation: crudele, e di tutte le colpe pigli vendetta, come ...
---
Original: Non d' altra forza d' animo fue ornato Ponzio Aufi...
Translation: Ponzio Aufidiano, romano cavaliere, fu ornato di u...
---


100%|██████████| 3/3 [01:37<00:00, 32.56s/it]



Sample results:
Original: quella guerra ben fatta l' opera perché etc. Et da...
Translation: Quella guerra ben fatta l'opera perché etc. Et dal...
---
Original: crudele, e di tutte le colpe pigli vendetta, come ...
Translation: Brutale, e di tutte le colpe prendere la vendetta,...
---
Original: Non d' altra forza d' animo fue ornato Ponzio Aufi...
Translation: Non c'è un'altra forza d'animo che fu ornato Ponzi...
---


100%|██████████| 3/3 [03:26<00:00, 68.83s/it]


Sample results:
Original: quella guerra ben fatta l' opera perché etc. Et da...
Translation: Quella guerra ben fatta è l'opera perché... Ecco, ...
---
Original: crudele, e di tutte le colpe pigli vendetta, come ...
Translation: Crudele, e di tutte le colpe piti vendetta, come d...
---
Original: Non d' altra forza d' animo fue ornato Ponzio Aufi...
Translation: Non d' altra forza d' animo fu decorato Ponzio Auf...
---


In [9]:
# For each prompt, translate all sentences and add as a new column in df
for idx, prompt in enumerate(prompts):
    col_name = f"Modern_Italian_{idx}"
    df[col_name] = translate(df, prompt)

100%|██████████| 97/97 [1:36:06<00:00, 59.44s/it] 


## Saving the Results

We'll save the original sentences along with their translations to a new CSV file.

In [10]:
output_file = 'qwen3/qwen3_translations.csv'
df.to_csv(output_file, index=False)

print(f"Saved translations to {output_file}")

Saved translations to qwen3/qwen3_translations.csv


# Evaluation (Zero-Shot & Instruction-Style)

In [21]:
# Read the saved translations file
translations_df = pd.read_csv('qwen3/qwen3_translations.csv')

# Show the first few rows
print("\nFirst 5 rows:")
translations_df.head()


First 5 rows:


,Author,Date,Region,Sentence,Modern_Italian_0,Modern_Italian_1,Modern_Italian_2,Modern_Italian_3
0,Brunetto Latini,1260-61,fior.,quella guerra ben fatta l' opera perché etc. E...,Quella guerra ben fatta l'opera perché etc. E ...,Quella guerra ben fatta l'opera perché etc. E ...,Quella guerra ben fatta l'opera perché etc. Et...,Quella guerra ben fatta l'opera perché... E da...
1,Bono Giamboni,1292,fior.,"crudele, e di tutte le colpe pigli vendetta, c...","crudele, e di tutte le colpe pigli vendetta, c...","crudele, e di tutte le colpe pigli vendetta, c...","Cruel, e prenda vendetta su tutte le colpe, co...","crudele, e di tutte le colpe prendi vendetta, ..."
2,Valerio Massimo (red. V1,1336,fior.,Non d' altra forza d' animo fue ornato Ponzio ...,Non d' altra forza d' animo fu ornato Ponzio A...,Non di un'altra forza d' animo fu ornato Ponzi...,Non un'altra forza d' animo fu ornato Ponzio A...,Non di un'altra forza d'animo fu ornato Ponzio...
3,Lucano volg. (ed. Marinoni),1330/40,prat.,Se questo piace a tutti e se 'l tempo hae biso...,Se questo piace a tutti e se 'l tempo hae biso...,Se questo piace a tutti e se 'l tempo hae biso...,Se questo piace a tutti e se 'l tempo hae biso...,Se questo piace a tutti e se 'l tempo hae biso...
4,Brunetto Latini,1260-61,fior.,Officio di questa arte pare che sia dicere app...,L’officio di questa arte sembra che sia dire a...,L'ufficio di questa arte sembra essere dire es...,L'officio di questa arte sembra essere dire in...,Officio di questa arte pare che sia dicere app...


In [22]:
import os
from dotenv import load_dotenv
from google import genai
import time

load_dotenv()
api_key = os.getenv("GEMINI_API_KEY")
client = genai.Client(api_key=api_key)

def evaluate_translation(row, translation_column, without_context = False) -> int:
    criteria = (
        "1. Completely unacceptable translation: the translation has no pertinence with the original meaning, the generated sentence is either gibberish or something that makes no sense.\n"
        "2. Severe semantic errors, omissions or substantial add ons on the original sentence. The errors are of semantic and syntactic nature. It’s still something no human would ever write.\n"
        "3. Partially wrong translation, the translation is lackluster, it contains errors, but are mostly minor errors, like typos, or small semantic errors.\n"
        "4. Good translation. The translation is mostly right, substantially faithful to the original text, but the style does not perfectly match the original sentence, still fluent and comprehensible, and could semantically acceptable.\n"
        "5. Perfect translation. The translation is accurate, fluent, complete and coherent. It retained the original meaning as much as it could."
    )

    if without_context:
        # Prompt without context
        prompt = (
            f"Rate the following translation on a scale of 1-5 based on these criteria:\n"
            f"{criteria}\n\n"
            f"Original sentence:\n\"{row['Sentence']}\"\n\n"
            f"Translated sentence:\n\"{row[translation_column]}\"\n\n"
            f"Provide only the rating (1-5)."
        )
    else:
        # Prompt with context
        prompt = (
            f"Rate this translation on a scale of 1-5 based on these criteria:\n"
            f"{criteria}\n\n"
            "Context:\n"
            f" • Author: {row['Author']}\n"
            f" • Date: {row['Date']}\n"
            f" • Region: {row['Region']}\n\n"
            f"Original sentence:\n\"{row['Sentence']}\"\n\n"
            f"Translated into Modern Italian:\n\"{row[translation_column]}\"\n\n"
            "Provide only the rating (1-5)."
        )
    resp = client.models.generate_content(
        model="gemini-2.0-flash",
        contents=prompt
    )
    try:
        return int(resp.text.strip())
    except ValueError:
        return None

In [23]:
def evaluate_translations_for_column(col: str, translations_df: pd.DataFrame, rate_limit_seconds: float = 4.5, filename = 'qwen3/qwen3_translations_with_eval.csv') -> pd.DataFrame:
    print(f"\nEvaluating translations for {col}...")
    
    # With context
    ratings = []
    for _, row in tqdm(translations_df.iterrows(), total=len(translations_df), desc=f"Evaluating {col} with context"):
        rating = evaluate_translation(row, col)
        ratings.append(rating)
        time.sleep(rate_limit_seconds)
    
    translations_df[f'{col}_gemini_eval'] = ratings
    
    # Without context
    ratings_no_context = []
    for _, row in tqdm(translations_df.iterrows(), total=len(translations_df), desc=f"Evaluating {col} without context"):
        rating = evaluate_translation(row, col, without_context=True)
        ratings_no_context.append(rating)
        time.sleep(rate_limit_seconds)
    
    translations_df[f'{col}_gemini_eval_no_context'] = ratings_no_context
    
    # Print summary statistics for this column
    print(f"\nEvaluation summary for {col}:")
    print(f"Average rating with context: {translations_df[f'{col}_gemini_eval'].mean():.2f}")
    print(f"Average rating without context: {translations_df[f'{col}_gemini_eval_no_context'].mean():.2f}")
    print(f"Difference: {(translations_df[f'{col}_gemini_eval'] - translations_df[f'{col}_gemini_eval_no_context']).mean():.2f}")
    
    # Save intermediate results
    translations_df.to_csv(filename, index=False)
    
    return translations_df

## Evaluating Translations from Prompt 0 (Role-based)

In [24]:
translations_df = evaluate_translations_for_column('Modern_Italian_0', translations_df)


Evaluating translations for Modern_Italian_0...


Evaluating Modern_Italian_0 without context: 100%|██████████| 97/97 [07:51<00:00,  4.86s/it]


Evaluation summary for Modern_Italian_0:
Average rating with context: 4.81
Average rating without context: 4.85
Difference: -0.03


## Evaluating Translations from Prompt 1 (No Role)

In [27]:
translations_df = evaluate_translations_for_column('Modern_Italian_1', translations_df)


Evaluating translations for Modern_Italian_1...


Evaluating Modern_Italian_1 without context: 100%|██████████| 97/97 [07:51<00:00,  4.86s/it]


Evaluation summary for Modern_Italian_1:
Average rating with context: 4.61
Average rating without context: 4.61
Difference: 0.00


## Evaluating Translations from Prompt 2 (With Context)

In [25]:
translations_df = evaluate_translations_for_column('Modern_Italian_2', translations_df)


Evaluating translations for Modern_Italian_2...


Evaluating Modern_Italian_2 without context: 100%|██████████| 97/97 [07:51<00:00,  4.86s/it]


Evaluation summary for Modern_Italian_2:
Average rating with context: 4.61
Average rating without context: 4.65
Difference: -0.04


## Evaluating Translations from Prompt 3 (Italian)

In [29]:
translations_df = evaluate_translations_for_column('Modern_Italian_3', translations_df)


Evaluating translations for Modern_Italian_3...


Evaluating Modern_Italian_3 without context: 100%|██████████| 97/97 [07:51<00:00,  4.86s/it]


Evaluation summary for Modern_Italian_3:
Average rating with context: 4.26
Average rating without context: 4.30
Difference: -0.04


# Translate the Sentences (Few-Shot)

In [2]:
# Load dataset
df = pd.read_csv("dataset.csv")
df.head()

,Author,Date,Region,Sentence
0,Brunetto Latini,1260-61,fior.,quella guerra ben fatta l' opera perché etc. E...
1,Bono Giamboni,1292,fior.,"crudele, e di tutte le colpe pigli vendetta, c..."
2,Valerio Massimo (red. V1,1336,fior.,Non d' altra forza d' animo fue ornato Ponzio ...
3,Lucano volg. (ed. Marinoni),1330/40,prat.,Se questo piace a tutti e se 'l tempo hae biso...
4,Brunetto Latini,1260-61,fior.,Officio di questa arte pare che sia dicere app...


In [7]:
few_shot_prompts = [
    [
        {"role": "system", "content": "You are an expert in historical Italian linguistics. Only provide the translation in your response, nothing else."},
        {"role": "user", "content": "Translate this archaic Italian sentence to modern Italian:\n… incontente mandà per li diti gotti, a li quae dolcementi parlando procurava cum doçe parole mitigar la lor aspreça"},
        {"role": "assistant", "content": "… subito mandò a chiamare quei suddetti Goti e, parlando con dolcezza, cercava con parole gentili di mitigare la loro durezza"},
        {"role": "user", "content": "Translate this archaic Italian sentence to modern Italian:\n… en leto agrevò de infirmitè ma de sana mente e de bone volontà e dretamentre parlando, no se voiando partir de questo mondo senza testamento"},
        {"role": "assistant", "content": "… si aggravò a letto per la malattia, ma con mente lucida e buona volontà e, parlando rettamente, non voleva lasciare questo mondo senza fare testamento"},
        {"role": "user", "content": "Translate this archaic Italian sentence to modern Italian:\n{sentence}"}
    ],
    [
        {"role": "system", "content": "Only provide the translation in your response, nothing else."},
        {"role": "user", "content": "Translate this archaic Italian sentence to modern Italian:\n… incontente mandà per li diti gotti, a li quae dolcementi parlando procurava cum doçe parole mitigar la lor aspreça"},
        {"role": "assistant", "content": "… subito mandò a chiamare quei suddetti Goti e, parlando con dolcezza, cercava con parole gentili di mitigare la loro durezza"},
        {"role": "user", "content": "Translate this archaic Italian sentence to modern Italian:\n… en leto agrevò de infirmitè ma de sana mente e de bone volontà e dretamentre parlando, no se voiando partir de questo mondo senza testamento"},
        {"role": "assistant", "content": "… si aggravò a letto per la malattia, ma con mente lucida e buona volontà e, parlando rettamente, non voleva lasciare questo mondo senza fare testamento"},
        {"role": "user", "content": "Translate this archaic Italian sentence to modern Italian:\n{sentence}"}
    ],
    [
        {"role": "system", "content": "Only provide the translation in your response, nothing else."},
        {"role": "user", "content": "Consider the following historical context:\nAuthor: anonimo\nTime period: XIV secolo\nGeographical region: Italia centro-settentrionale\n\nHere is an archaic Italian sentence to translate to modern Italian:\n… incontente mandà per li diti gotti, a li quae dolcementi parlando procurava cum doçe parole mitigar la lor aspreça"},
        {"role": "assistant", "content": "… subito mandò a chiamare quei suddetti Goti e, parlando con dolcezza, cercava con parole gentili di mitigare la loro durezza"},
        {"role": "user", "content": "Consider the following historical context:\nAuthor: anonimo\nTime period: XIV secolo\nGeographical region: Italia nord-orientale\n\nHere is an archaic Italian sentence to translate to modern Italian:\n… en leto agrevò de infirmitè ma de sana mente e de bone volontà e dretamentre parlando, no se voiando partir de questo mondo senza testamento"},
        {"role": "assistant", "content": "… si aggravò a letto per la malattia, ma con mente lucida e buona volontà e, parlando rettamente, non voleva lasciare questo mondo senza fare testamento"},
        {"role": "user", "content": "Consider the following historical context:\nAuthor: {author}\nTime period: {date}\nGeographical region: {region}\n\nHere is an archaic Italian sentence to translate to modern Italian:\n{sentence}"}
    ],
    [
        {"role": "system", "content": "Sei un esperto di linguistica storica italiana. Rispondi con SOLO la traduzione."},
        {"role": "user", "content": "Traduci questa frase in italiano antico (1300) in italiano moderno:\n… incontente mandà per li diti gotti, a li quae dolcementi parlando procurava cum doçe parole mitigar la lor aspreça"},
        {"role": "assistant", "content": "… subito mandò a chiamare quei suddetti Goti e, parlando con dolcezza, cercava con parole gentili di mitigare la loro durezza"},
        {"role": "user", "content": "Traduci questa frase in italiano antico (1300) in italiano moderno:\n… en leto agrevò de infirmitè ma de sana mente e de bone volontà e dretamentre parlando, no se voiando partir de questo mondo senza testamento"},
        {"role": "assistant", "content": "… si aggravò a letto per la malattia, ma con mente lucida e buona volontà e, parlando rettamente, non voleva lasciare questo mondo senza fare testamento"},
        {"role": "user", "content": "Traduci questa frase in italiano antico (1300) in italiano moderno:\n{sentence}"}
    ]
]

In [8]:
for idx, prompt in enumerate(few_shot_prompts):
    col_name = f"Modern_Italian_{idx}"
    df[col_name] = translate(df, prompt)

100%|██████████| 97/97 [1:31:36<00:00, 56.66s/it]


In [9]:
output_file = 'qwen3/qwen3_translations_fewshot.csv'
df.to_csv(output_file, index=False)

print(f"Saved translations to {output_file}")

Saved translations to qwen3/qwen3_translations_fewshot.csv


# Evaluation (Few-Shot)

In [6]:
translations_df = pd.read_csv('qwen3/qwen3_translations_fewshot.csv')

translations_df.head()

,Author,Date,Region,Sentence,Modern_Italian_0,Modern_Italian_1,Modern_Italian_2,Modern_Italian_3
0,Brunetto Latini,1260-61,fior.,quella guerra ben fatta l' opera perché etc. E...,Quella guerra ben fatta l'opera perché etc. Da...,"E, dall' altra parte, Aiaces era un cavaliere ...","Quella guerra fu ben fatta, l' opera perché......",Quella guerra ben fatta fu l'opera perché... E...
1,Bono Giamboni,1292,fior.,"crudele, e di tutte le colpe pigli vendetta, c...","Cruel, e prenda la vendetta su tutti i peccati...","Brutale, e di tutte le colpe prendere vendetta...","Cruel, e prendere vendetta per tutte le colpe,...","Cruel, e di tutte le colpe pigli vendetta, com..."
2,Valerio Massimo (red. V1,1336,fior.,Non d' altra forza d' animo fue ornato Ponzio ...,"Non aveva altra forza d'animo, Ponzio Aufidian...","Non aveva alcun altro potere d'anima, il caval...",Non fu ornato da alcun'altra forza d'animo il ...,Non d' altra forza d'anima fu ornato Ponzio Au...
3,Lucano volg. (ed. Marinoni),1330/40,prat.,Se questo piace a tutti e se 'l tempo hae biso...,Se questo piace a tutti e se il tempo ha bisog...,"Se questo piace a tutti, e se il tempo ha biso...",Se questo piace a tutti e se il tempo ha bisog...,Se questo piace a tutti e se il tempo ha bisog...
4,Brunetto Latini,1260-61,fior.,Officio di questa arte pare che sia dicere app...,L'officio di questa arte sembra essere dire sp...,Ufficio di questa arte pare che sia dicere app...,L'officio di questa arte sembra che sia dire i...,L'officio di questa arte pare che sia dire esp...


## Evaluating Translations from Prompt 0 (Role-based)

In [10]:
translations_df = evaluate_translations_for_column('Modern_Italian_0', translations_df, filename='qwen3/qwen3_translations_fewshot_with_eval.csv')


Evaluating translations for Modern_Italian_0...


Evaluating Modern_Italian_0 without context: 100%|██████████| 97/97 [07:50<00:00,  4.85s/it]


Evaluation summary for Modern_Italian_0:
Average rating with context: 4.33
Average rating without context: 4.18
Difference: 0.15


## Evaluating Translations from Prompt 1 (No Role)

In [9]:
translations_df = evaluate_translations_for_column('Modern_Italian_1', translations_df, filename='qwen3/qwen3_translations_fewshot_with_eval.csv')


Evaluating translations for Modern_Italian_1...


Evaluating Modern_Italian_1 without context: 100%|██████████| 97/97 [07:52<00:00,  4.88s/it]


Evaluation summary for Modern_Italian_1:
Average rating with context: 4.24
Average rating without context: 4.16
Difference: 0.07


## Evaluating Translations from Prompt 2 (With Context)

In [11]:
translations_df = evaluate_translations_for_column('Modern_Italian_2', translations_df, filename='qwen3/qwen3_translations_fewshot_with_eval.csv')


Evaluating translations for Modern_Italian_2...


Evaluating Modern_Italian_2 without context: 100%|██████████| 97/97 [07:53<00:00,  4.88s/it]


Evaluation summary for Modern_Italian_2:
Average rating with context: 4.21
Average rating without context: 4.03
Difference: 0.18


## Evaluating Translations from Prompt 3 (Italian)

In [20]:
translations_df = evaluate_translations_for_column('Modern_Italian_3', translations_df, filename='qwen3/qwen3_translations_fewshot_with_eval.csv')


Evaluating translations for Modern_Italian_3...


Evaluating Modern_Italian_3 without context: 100%|██████████| 97/97 [07:53<00:00,  4.88s/it]


Evaluation summary for Modern_Italian_3:
Average rating with context: 4.31
Average rating without context: 4.21
Difference: 0.10
